In [ ]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
import torch
import time
import numpy as np
import torch.nn as nn

from tqdm import tqdm
from collections import defaultdict
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

def quantize_per_vector(tensor, mantissa_bits, dim):
    max_quant_val = (2**mantissa_bits) - 1

    # Calculate scaling factors (sf) per vector along the specified dimension
    max_abs_vals = torch.max(torch.abs(tensor), dim=dim, keepdim=True).values
    sf = torch.where(max_abs_vals > 0, max_abs_vals / max_quant_val, 1.0)

    # Quantize by dividing by the scaling factor (broadcasting applies it correctly)
    quantized_tensor = torch.round(tensor / sf)

    return quantized_tensor, sf


def add_adc_noise(tensor, mean, sigma, lsb):
    if sigma <= 0:
        return tensor
    # Sample noise from a Gaussian distribution in the voltage domain
    noise_in_volts = torch.randn(tensor.size(), device=tensor.device) * sigma + mean
    # Convert the voltage noise into an equivalent integer error by scaling by the LSB
    integer_noise = torch.round(noise_in_volts / lsb)
    return tensor + integer_noise

class SimulatedFFN(nn.Module):

    def __init__(self, original_mlp, tile_shape=(32, 40), noise_mean=0.0, noise_sigma=0.001):
        super().__init__()
        # Extract weights from the original FFN layer
        self.gate_proj = original_mlp.gate_proj
        self.up_proj = original_mlp.up_proj
        self.down_proj = original_mlp.down_proj

        # Simulation parameters
        v_min = -0.5
        v_max = 0.5
        levels = 256
        fsr = v_max - v_min
        self.lsb = fsr / levels
        self.noise_mean = noise_mean
        self.noise_sigma = noise_sigma
        self.tile_shape = tile_shape

    @torch.compile(@torch.compile(options={"triton.cudagraphs": True}, fullgraph=True))
    def tiled_mat_mul_vec(self, input_tensor, weight_tensor):
        weight_tensor = weight_tensor.T
        m, k = input_tensor.shape
        _, n = weight_tensor.shape
        tile_m, block_size = self.tile_shape
        tile_n = tile_m  # Use square tiles for output

        # Pad tensors to be divisible by tile dimensions
        pad_m = (tile_m - m % tile_m) % tile_m
        pad_k = (block_size - k % block_size) % block_size
        pad_n = (tile_n - n % tile_n) % tile_n

        inp_pad = nn.functional.pad(input_tensor, (0, pad_k, 0, pad_m))
        #w_pad = nn.functional.pad(weight_tensor.T, (0, pad_n, 0, pad_k)).T
        w_pad = nn.functional.pad(weight_tensor, (0, pad_n, 0, pad_k))
        m_pad, k_pad = inp_pad.shape
        _, n_pad = w_pad.shape

        # Create unfolded views (tiles) of the input and weight tensors
        inp_tiles = inp_pad.unfold(0, tile_m, tile_m).unfold(1, block_size, block_size)
        num_m_tiles, num_k_tiles, _, _ = inp_tiles.shape

        w_tiles = w_pad.unfold(0, block_size, block_size).unfold(1, tile_n, tile_n)
        _, num_n_tiles, _, _ = w_tiles.shape

        # Quantize each input tile (32x40) per-row
        inp_quant_tiles, inp_sf_tiles = quantize_per_vector(inp_tiles, mantissa_bits=7, dim=3)

        # Quantize each weight tile (40x32) per-column
        w_quant_tiles, w_sf_tiles = quantize_per_vector(w_tiles, mantissa_bits=2, dim=2)

        # Perform matmul on integer mantissas for each tile.
        # This computes the accumulated integer value for each element of the tile's output.
        # Unsqueeze to enable broadcasting over all tiles at once.
        accum_mantissas = torch.matmul(
            inp_quant_tiles.unsqueeze(2).to(torch.bfloat16),
            w_quant_tiles.unsqueeze(0).to(torch.bfloat16)
        )

        # Apply noise to the accumulated mantissa products of each tile output.
        noisy_accum_mantissas = add_adc_noise(accum_mantissas, self.noise_mean,
                                              self.noise_sigma, self.lsb)

        # Dequantize the noisy results by applying the tile-specific row and column scalars.
        # Unsqueeze scalars for correct broadcasting over the (m_tile, k_tile, n_tile, 32, 32) result.
        combined_sf = inp_sf_tiles.unsqueeze(2) * w_sf_tiles.unsqueeze(0)
        dequantized_tiles = noisy_accum_mantissas * combined_sf

        # Accumulate the dequantized tile outputs with perfect precision.
        output_tiles = torch.sum(dequantized_tiles, dim=1).to(input_tensor.dtype)

        # Reshape output tiles back to a 2D matrix
        out_pad = output_tiles.permute(0, 2, 1, 3).contiguous().view(m_pad, n_pad)

        return out_pad[:m, :n]


    def forward(self, x):
        batch_size, seq_len, hidden_dim = x.shape
        x_2d = x.view(-1, hidden_dim)

        # Gating is done in high precision
        gated_states = self.gate_proj(x_2d)
        gated_states = nn.functional.gelu(gated_states)

        # Up Projection uses simulated matrix multiplication
        up_states = self.tiled_mat_mul_vec(x_2d, self.up_proj.weight)
        activated_states = gated_states * up_states

        # Down Projection uses simulated matrix multiplication
        output_states_2d = self.tiled_mat_mul_vec(activated_states, self.down_proj.weight)
        return output_states_2d.view(batch_size, seq_len, -1)

def replace_ffn_with_simulation(model, noise_mean, noise_sigma):
    for layer in model.model.layers:
        original_mlp = layer.mlp
        layer.mlp = SimulatedFFN(original_mlp, noise_mean=noise_mean, noise_sigma=noise_sigma)
    print("Replacement complete.")
    return model

def format_mmlu_prompt(dev_samples, test_sample, subject_name, k_shot):
    prompt = f"The following are multiple choice questions (with answers) about {subject_name}.\n\n"

    # Add few-shot examples if k_shot > 0
    if k_shot > 0:
        for i in range(k_shot):
            question = dev_samples[i]['question']
            choices_str = "\n".join([f"{chr(65+j)}. {dev_samples[i]['choices'][j]}" for j in range(len(dev_samples[i]['choices']))])
            prompt += f"Question: {question}\n{choices_str}\nAnswer: {chr(65+dev_samples[i]['answer'])}\n\n"

    # Add the actual test question
    question = test_sample['question']
    choices_str = "\n".join([f"{chr(65+j)}. {test_sample['choices'][j]}" for j in range(len(test_sample['choices']))])
    prompt += f"Question: {question}\n{choices_str}\nAnswer:"
    return prompt, chr(65 + test_sample['answer'])

@torch.no_grad()
def evaluate_mmlu(model, tokenizer, device, k_shot=5):
    print(f"Loading MMLU dataset for {k_shot}-shot evaluation...")
    mmlu_dataset = load_dataset("cais/mmlu", "all")
    dev_examples_by_subject = defaultdict(list)
    if k_shot > 0:
        print("Grouping Few-Shot Examples by subject")
        for example in tqdm(mmlu_dataset["dev"], desc="Processing Dev Split"):
            dev_examples_by_subject[example["subject"]].append(example)

    subjects = sorted(list(set(mmlu_dataset["test"]["subject"])))
    results = {}
    total_correct = 0
    total_all = 0

    print("Starting MMLU evaluation...")
    for subject in tqdm(subjects, desc="Evaluating Subjects"):
        print(f"Running for Subject {subject}")
        subject_correct = 0
        subject_test_set = mmlu_dataset["test"].filter(lambda example: example["subject"] == subject)
        subject_dev_samples = dev_examples_by_subject[subject] if k_shot > 0 else None

        for test_sample in subject_test_set:
            # Prepare sample for the prompt
            prompt, correct_answer = format_mmlu_prompt(subject_dev_samples, test_sample, subject.replace("_", " "), k_shot=k_shot)
            inputs = tokenizer(prompt, return_tensors="pt").to(device)

            outputs = model(**inputs)
            logits = outputs.logits[0, -1]
            choice_tokens = [tokenizer.encode(f" {chr(65+j)}", add_special_tokens=False)[-1] for j in range(len(test_sample["choices"]))]
            choice_logits = logits[choice_tokens]
            prediction_idx = torch.argmax(choice_logits).item()
            prediction = chr(65 + prediction_idx)

            if prediction == correct_answer:
                subject_correct += 1
        torch.cuda.empty_cache()
        accuracy = subject_correct / len(subject_test_set)
        results[subject] = accuracy
        print(f"  Subject: {subject}, Accuracy: {accuracy:.4f}")
        total_correct += subject_correct
        total_all += len(subject_test_set)

    overall_accuracy = total_correct / total_all
    results["overall_accuracy"] = overall_accuracy
    print(f"\nOverall MMLU Accuracy ({k_shot}-shot): {overall_accuracy:.4f}")
    return results

In [ ]:
class Logger:
    def __init__(self, filepath, stream):
        self.stream = stream
        # Open the file with line buffering (buffering=1) for real-time output
        self.log = open(filepath, 'a', buffering=1)

    def write(self, message):
        self.stream.write(message)
        self.log.write(message)

    def flush(self):
        # This flush method is needed for compatibility with some libraries.
        self.stream.flush()
        self.log.flush()

    def __del__(self):
        # Ensure the log file is closed when the object is destroyed.
        self.log.close()

In [ ]:
MODEL_ID = "google/gemma-2-2b"
out_path = "./drive/Shareddrives/MXPTQ"

NOISE_MEAN = 0.0
NOISE_SIGMA = 0.001

K_SHOT = 5

device = "cuda" if torch.cuda.is_available() else "cpu"

log_file_path = f"{out_path}/quant_hw_output.log"

original_stdout = sys.stdout
original_stderr = sys.stderr
sys.stdout = Logger(log_file_path, original_stdout)
sys.stderr = Logger(log_file_path, original_stderr)

try:
    print(f"Using device: {device}")
    print("Loading base model and tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map=device,
    )
    print("Running Quantization with HW Noise Run")
    quant_hw_model = replace_ffn_with_simulation(model, noise_mean=NOISE_MEAN, noise_sigma=NOISE_SIGMA)
    quant_hw_results = evaluate_mmlu(quant_hw_model, tokenizer, device, k_shot=K_SHOT)
    torch.save(quant_hw_results, f"{out_path}/quant_hw_results.pt")
finally:
    # --- Restore original stdout and stderr ---
    # This block will run even if the code in the 'try' block fails.
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    print("--- Logging restored to notebook output ---")

Using device: cuda
Loading base model and tokenizer...


tokenizer_config.json:   0%|          | 0.00/46.4k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Evaluating Subjects:  98%|#########8| 56/57 [5:34:13<05:50, 350.39s/it]

  Subject: virology, Accuracy: 0.3976
Running for Subject world_religions


Filter:   0%|          | 0/14042 [00:00<?, ? examples/s]

Evaluating Subjects: 100%|##########| 57/57 [5:35:51<00:00, 353.54s/it]

  Subject: world_religions, Accuracy: 0.6959

Overall MMLU Accuracy (5-shot): 0.4729
--- Logging restored to notebook output ---
